In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_parquet('../data/raw/raw_prices.parquet')
df.shape

(2809, 20)

In [4]:
df_log = np.log((df)/(df.shift(1)))
df_log.dropna(inplace=True)
df_log.tail()

Ticker,ADI,AMD,AVGO,CRUS,INTC,LSCC,MCHP,MPWR,MRVL,MU,NVDA,NXPI,ON,QCOM,QRVO,SLAB,SMTC,STM,SWKS,TXN
Date,,,,,,,,,,,,,,,,,,,,
2026-02-27,0.004055,-0.017183,-0.006706,-0.027261,0.003294,-0.016903,-0.004412,-0.032196,0.029820,-0.007706,-0.042538,-0.022734,-0.024957,-0.022435,0.005322,-0.000929,-0.002988,-0.022651,-0.000503,-0.002449
2026-03-02,-0.009545,-0.007973,-0.002287,0.013444,-0.002415,0.051962,-0.004431,-0.000140,-0.010212,0.000727,0.029418,-0.009961,0.000000,-0.009386,-0.011892,0.001856,0.065217,-0.003875,-0.010970,-0.010855
2026-03-03,-0.035698,-0.039382,-0.015743,-0.033051,-0.054189,-0.074597,-0.040088,-0.061554,-0.042312,-0.083320,-0.013406,-0.043233,-0.047122,-0.020777,-0.007966,-0.006217,-0.065772,-0.057158,-0.013324,-0.034671
2026-03-04,0.007406,0.056603,0.011689,-0.004128,0.055946,-0.019989,-0.021092,0.022684,0.007455,0.054059,0.016470,0.005190,-0.014133,0.009941,-0.022523,-0.000295,-0.004669,0.052668,-0.032687,-0.001383
2026-03-05,-0.035133,-0.013051,0.046879,-0.025949,0.008085,-0.005691,-0.030356,-0.018903,-0.031348,-0.009325,0.001638,-0.027124,-0.027235,-0.011755,-0.000629,-0.001622,-0.030201,-0.000600,0.003547,-0.022031


In [32]:
total_days = len(df_log.index)
valid_days_series = total_days - df_log.isna().sum()
pct_missing_values = 1 - (valid_days_series/total_days)
mean_series = df_log.mean()
std_series = df_log.std()
min_series = df_log.min()
max_series = df_log.max()

status = (pct_missing_values < 0.02) & (min_series > -0.6) & (max_series < 0.8) & (std_series < 3*std_series.median())


summary_df = pd.DataFrame({
    'Valid Days': valid_days_series,
    'Pct. Missing Days': pct_missing_values,
    'Mean Return':mean_series,
    'Std Return':std_series,
    'Min Return':min_series,
    'Max Return':max_series
})

summary_df['Status'] = np.where(status, 'Ticker Check - PASS', 'Ticker Check - FAIL')

summary_df

,Valid Days,Pct. Missing Days,Mean Return,Std Return,Min Return,Max Return,Status
Ticker,,,,,,,
ADI,2808,0.0,0.000716,0.020227,-0.181700,0.168794,Ticker Check - PASS
AMD,2808,0.0,0.001536,0.036555,-0.277456,0.420617,Ticker Check - PASS
AVGO,2808,0.0,0.001348,0.024550,-0.222055,0.218594,Ticker Check - PASS
CRUS,2808,0.0,0.000618,0.025920,-0.188414,0.176238,Ticker Check - PASS
INTC,2808,0.0,0.000178,0.025349,-0.301896,0.205151,Ticker Check - PASS
LSCC,2808,0.0,0.000918,0.031497,-0.190649,0.259162,Ticker Check - PASS
MCHP,2808,0.0,0.000476,0.025147,-0.226830,0.239422,Ticker Check - PASS
MPWR,2808,0.0,0.001139,0.028218,-0.226653,0.210502,Ticker Check - PASS
MRVL,2808,0.0,0.000626,0.030339,-0.220817,0.280836,Ticker Check - PASS


In [ ]:
mask_10 = df_log.abs() > 0.1
mask_25 = df_log.abs() > 0.25

extreme_10 = mask_10.sum()
extreme_25 = mask_25.sum()

breaches_per_day = mask_10.sum(axis=1)
stress_dates = breaches_per_day[breaches_per_day >=3].sort_values(ascending = False)

worst_return_per_ticker = df_log.min()
best_return_per_ticker = df_log.max()

mask_25_non_stress = mask_25[mask_25.index.isin(stress_dates.index)]
sus_ticker = mask_25_non_stress[mask_25_non_stress > 0].index.tolist()

status = ('Extremes Check - PASS' if (len(sus_ticker) == 0) else 'Extreme Check - FAIL')

['AMD', 'INTC', 'LSCC', 'MRVL', 'NVDA', 'ON', 'QRVO', 'SLAB', 'SMTC', 'SWKS']

In [58]:
mask_25_non_stress = mask_25[~mask_25.index.isin(stress_dates.index)]
mask_25_non_stress.any().index.tolist()

['ADI',
 'AMD',
 'AVGO',
 'CRUS',
 'INTC',
 'LSCC',
 'MCHP',
 'MPWR',
 'MRVL',
 'MU',
 'NVDA',
 'NXPI',
 'ON',
 'QCOM',
 'QRVO',
 'SLAB',
 'SMTC',
 'STM',
 'SWKS',
 'TXN']

In [69]:
sus_ticker = mask_25_non_stress.any()[mask_25_non_stress.any()].index.tolist()
sus_ticker

['AMD', 'INTC', 'LSCC', 'MRVL', 'NVDA', 'QRVO', 'SMTC', 'SWKS']